In [1]:
import pandas as pd
from collections import defaultdict, Counter
import re
import spacy
import random

nlp = spacy.load('en_core_web_sm')

In [2]:
df = pd.read_csv('tweets.csv')
df.head()

,source,text,created_at,retweet_count,favorite_count,is_retweet,id_str
0,Twitter for iPhone,LOSER! https://t.co/p5imhMJqS1,05-18-2020 14:55:14,32295,135445,False,1262396333064892416
1,Twitter for iPhone,Most of the money raised by the RINO losers of...,05-05-2020 18:18:26,19706,82425,False,1257736426206031874
2,Twitter for iPhone,....because they don’t know how to win and the...,05-05-2020 04:46:34,12665,56868,False,1257532112233803782
3,Twitter for iPhone,....lost for Evan “McMuffin” McMullin (to me)....,05-05-2020 04:46:34,13855,62268,False,1257532114666508291
4,Twitter for iPhone,....get even for all of their many failures. Y...,05-05-2020 04:46:33,8122,33261,False,1257532110971318274


In [11]:
def spacy_tokenize(text):
    doc = nlp(text.lower())
    return [token.text for token in doc if not token.is_space and not token.is_punct]

df['tokens'] = df['text'].apply(spacy_tokenize)
df.head()

,source,text,created_at,retweet_count,favorite_count,is_retweet,id_str,tokens
0,Twitter for iPhone,LOSER! https://t.co/p5imhMJqS1,05-18-2020 14:55:14,32295,135445,False,1262396333064892416,"[loser, https://t.co/p5imhmjqs1]"
1,Twitter for iPhone,Most of the money raised by the RINO losers of...,05-05-2020 18:18:26,19706,82425,False,1257736426206031874,"[most, of, the, money, raised, by, the, rino, ..."
2,Twitter for iPhone,....because they don’t know how to win and the...,05-05-2020 04:46:34,12665,56868,False,1257532112233803782,"[because, they, do, n’t, know, how, to, win, a..."
3,Twitter for iPhone,....lost for Evan “McMuffin” McMullin (to me)....,05-05-2020 04:46:34,13855,62268,False,1257532114666508291,"[lost, for, evan, mcmuffin, mcmullin, to, me, ..."
4,Twitter for iPhone,....get even for all of their many failures. Y...,05-05-2020 04:46:33,8122,33261,False,1257532110971318274,"[get, even, for, all, of, their, many, failure..."


In [12]:
# Costruzione del modello di bigrammi
bigram_model = defaultdict(Counter)

for tokens in df['tokens']:
    for i in range(len(tokens) - 1):
        bigram_model[tokens[i]][tokens[i+1]] += 1

# Costruzione del modello di trigrammi
trigram_model = defaultdict(Counter)

for tokens in df['tokens']:
    for i in range(len(tokens) - 2):
        trigram_model[(tokens[i], tokens[i+1])][tokens[i+2]] += 1

print(bigram_model["his"])

Counter({'lover': 4, 'wife': 3, 'own': 2, 'phony': 2, 'dumb': 2, 'show': 2, 'political': 1, 'reputation': 1, 'seat': 1, 'supporters': 1, 'job': 1, 'ratings': 1, 'recent': 1, 'success': 1, 'experience': 1, 'friends': 1, 'stupidity': 1})


In [13]:
def generate_tweet_bi(seed, model, max_len=20):
    current_word = seed
    tweet = [current_word]
    
    for _ in range(max_len - 1):
        if current_word in model:
            next_word = random.choices(list(model[current_word].keys()), list(model[current_word].values()))[0]
            tweet.append(next_word)
            current_word = next_word
        else:
            break
    
    return ' '.join(tweet)

def generate_tweet_tri(seed_tuple, model, max_len=20):
    current_words = seed_tuple
    tweet = list(current_words)
    
    for _ in range(max_len - 2):
        if current_words in model:
            next_word = random.choices(list(model[current_words].keys()), list(model[current_words].values()))[0]
            tweet.append(next_word)
            current_words = (current_words[1], next_word)
        else:
            break
    
    return ' '.join(tweet)


In [15]:
# Generazione tweet usando il modello di bigrammi
seed_word = "people"  # Scegli un seed di partenza
tweet_bi = generate_tweet_bi(seed_word, bigram_model)
print(f"Tweet generato (Bigrammi): {tweet_bi}")

# Generazione tweet usando il modello di trigrammi
seed_tuple = ("make", "america")  # Scegli un seed di partenza
tweet_tri = generate_tweet_tri(seed_tuple, trigram_model)
print(f"Tweet generato (Trigrammi): {tweet_tri}")


Tweet generato (Bigrammi): people who the haters i like bullies i have always liked amp losers of the worst and losers say i
Tweet generato (Trigrammi): make america great again i say they will do anything to get rid of the losers do n't understand the
